# MLP — Exercício 2

## Exercise 2

[Enunciado](https://insper.github.io/ann-dl/2026.2/exercises/mlp/)

**Uso de IA:** Codex para elaboração da implementação em NumPy, revisão dos gradientes, gráficos e organização da análise.


In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Execução a partir da raiz do repositório ou da pasta do notebook.
code_dir = Path.cwd()
if not (code_dir / "exercicio_2.py").is_file():
    code_dir = Path.cwd() / "docs" / "exercises" / "mlp" / "code"
if not (code_dir / "exercicio_2.py").is_file():
    raise FileNotFoundError("Abra o notebook na pasta code ou na raiz do repositório.")
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))
from exercicio_2 import plot_data, plot_loss, plot_boundary

rng = np.random.default_rng(42)


### A — Generate the data

Foram geradas **1000 amostras com duas características**, usando os parâmetros do enunciado sem alterações. Os três grupos têm centros `[0, 0]`, `[3, 3]` e `[-3, -3]`, respectivamente, com 500, 250 e 250 amostras e desvio padrão `1.2`.

A expressão `y = (c > 0).astype(int)` mantém o grupo central na classe 0 e reúne os dois grupos afastados na classe 1. Assim, cada classe tem **500 amostras**.

A divisão usa `test_size=0.2`, `random_state=42` e `stratify=y`: **800 pontos no treino (400 de cada classe)** e **200 no teste (100 de cada classe)**. O teste foi reservado para a avaliação final. As características foram usadas na escala original; não houve ajuste de pré-processamento usando o teste.

A semente dos dados e da divisão é 42. O objeto `rng = np.random.default_rng(42)` é usado na inicialização dos pesos da rede.


In [2]:
centers = np.array([[0., 0.], [3., 3.], [-3., -3.]])
X, c = make_blobs(n_samples=[500, 250, 250], centers=centers,
                  cluster_std=1.2, random_state=42)
y = (c > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
plot_data(X, y)
print(f"Dados: {X.shape}; classes: {np.bincount(y)}")
print(f"Treino: {len(X_train)}; classes: {np.bincount(y_train)}")
print(f"Teste: {len(X_test)}; classes: {np.bincount(y_test)}")


Dados: (1000, 2); classes: [500 500]
Treino: 800; classes: [400 400]
Teste: 200; classes: [100 100]


![Figura 1 — Dados de classificação binária](../figures/fig1.png)


### B — Establish the baseline

A regressão logística foi ajustada apenas nos dados de treino, com `max_iter=1000` e `random_state=42`. Sua saída depende de uma combinação linear das duas características, portanto sua fronteira de decisão é uma reta.

A **acurácia no teste foi de 47,5%**, equivalente a **95 acertos em 200 amostras**. Esse resultado será comparado com a MLP.


In [3]:
baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)
baseline_accuracy = np.mean(baseline.predict(X_test) == y_test)
print(f"Acurácia do baseline: {baseline_accuracy:.1%}")


Acurácia do baseline: 47.5%


### C — Implement the MLP

A MLP foi implementada em NumPy com os tamanhos das camadas como argumento. Foi usada a arquitetura **2 → 8 → 1**, com uma camada oculta de oito neurônios `tanh` e uma saída `sigmoid`. Os biases começam em zero e os pesos usam inicialização de Xavier, com desvio padrão $\sqrt{2/(n_{\mathrm{entrada}}+n_{\mathrm{saída}})}$.

| Configuração | Valor |
|---|---|
| Arquitetura | 2 entradas, 8 neurônios ocultos, 1 saída |
| Parâmetros treináveis | 33 |
| Ativação oculta | tanh |
| Ativação de saída | sigmoid |
| Perda | Entropia cruzada binária média |
| Taxa de aprendizado | 0,1 |
| Épocas | 2000 |
| Lote | Todas as 800 amostras de treino |
| Inicialização | Xavier, `rng` com semente 42 |
| Limiar de classificação | 0,5 |

Essas configurações foram definidas antes da avaliação do teste. A acurácia é calculada diretamente como a proporção de previsões corretas.

#### Forward pass e perda

Neste exercício, as amostras são organizadas nas **linhas** das matrizes. Para cada camada $\ell$, os pesos têm dimensão entrada × saída e o bias é somado a cada amostra:

$$
\mathbf{A}^{(0)}=\mathbf{X},\qquad
\mathbf{Z}^{(\ell)}=\mathbf{A}^{(\ell-1)}\mathbf{W}^{(\ell)}+\mathbf{b}^{(\ell)}.
$$

Nas camadas ocultas, $\mathbf{A}^{(\ell)}=\tanh(\mathbf{Z}^{(\ell)})$.
Na saída binária, cada logit $u_i$ é convertido em probabilidade:

$$
p_i=\sigma(u_i)=\frac{1}{1+e^{-u_i}}.
$$

A perda média é:

$$
L=-\frac{1}{N}\sum_{i=1}^{N}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right].
$$

No código, a mesma perda é calculada pela expressão equivalente
$\frac{1}{N}\sum_i[\log(1+e^{u_i})-y_i u_i]$, usando `np.logaddexp(0, u)` para evitar problemas numéricos nas exponenciais e em `log(0)`.

#### Backward pass

Para cada probabilidade:

$$
\frac{\partial L}{\partial p_i}
=\frac{1}{N}\left[-\frac{y_i}{p_i}+\frac{1-y_i}{1-p_i}\right]
=\frac{p_i-y_i}{Np_i(1-p_i)}.
$$

Como $\partial p_i/\partial u_i=p_i(1-p_i)$, a regra da cadeia resulta em:

$$
\frac{\partial L}{\partial u_i}
=\frac{p_i-y_i}{Np_i(1-p_i)}p_i(1-p_i)
=\frac{p_i-y_i}{N}.
$$

Chamando a derivada em relação às pré-ativações de $\mathbf{\Delta}^{(\ell)}$, na última camada:

$$
\mathbf{\Delta}^{(L)}=\frac{\mathbf{P}-\mathbf{Y}}{N}.
$$

Para qualquer camada, os gradientes são:

$$
\frac{\partial L}{\partial\mathbf{W}^{(\ell)}}
=(\mathbf{A}^{(\ell-1)})^T\mathbf{\Delta}^{(\ell)},\qquad
\frac{\partial L}{\partial\mathbf{b}^{(\ell)}}
=\sum_{i=1}^{N}\mathbf{\Delta}_{i,:}^{(\ell)}.
$$

Para propagar a derivada à camada oculta anterior:

$$
\mathbf{\Delta}^{(\ell-1)}
=\left[\mathbf{\Delta}^{(\ell)}(\mathbf{W}^{(\ell)})^T\right]
\odot\left[1-(\mathbf{A}^{(\ell-1)})^2\right],
$$

onde $\odot$ indica multiplicação elemento a elemento. A divisão por $N$ aparece uma única vez, na derivada da saída.

#### Atualização e resultados

Todos os gradientes são calculados antes de atualizar os parâmetros:

$$
\mathbf{W}^{(\ell)}\leftarrow\mathbf{W}^{(\ell)}-\eta\frac{\partial L}{\partial\mathbf{W}^{(\ell)}},\qquad
\mathbf{b}^{(\ell)}\leftarrow\mathbf{b}^{(\ell)}-\eta\frac{\partial L}{\partial\mathbf{b}^{(\ell)}}.
$$

O treino registrou a perda inicial e a perda após cada atualização. Ela caiu de **0,8270 para 0,1134**. A MLP obteve **91,5% de acurácia no teste**, com **183 acertos em 200 pontos** — **44 pontos percentuais** acima do baseline.

A ativação da saída e a função que retorna a perda e sua derivada são argumentos da classe. Assim, os métodos `forward`, `backward` e `update` podem ser reutilizados com outros tamanhos de camadas e outras saídas.


In [4]:
"""MLP em NumPy, com número de camadas definido por layer_sizes.

As funções da saída e da perda são argumentos para permitir o reuso do
mesmo forward, backward e update em classificação binária ou multiclasse.
"""

import numpy as np


def sigmoid(z):
    """Sigmoid calculada por partes para evitar overflow nas exponenciais."""
    probabilities = np.empty_like(z, dtype=float)
    positive = z >= 0
    probabilities[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exponential = np.exp(z[~positive])
    probabilities[~positive] = exponential / (1.0 + exponential)
    return probabilities


def binary_cross_entropy(logits, probabilities, y):
    """Retorna a BCE média e sua derivada em relação aos logits.

    log(1 + exp(z)) - y*z é equivalente à BCE com sigmoid, mas evita
    log(0). A derivada (p-y)/N já inclui a média sobre as amostras.
    """
    targets = np.asarray(y, dtype=float).reshape(-1, 1)
    loss = np.mean(np.logaddexp(0.0, logits) - targets * logits)
    delta = (probabilities - targets) / len(targets)
    return float(loss), delta


class MLP:
    """Camadas ocultas tanh e saída/perda configuráveis, sem autodiff."""

    def __init__(self, layer_sizes, rng, output_activation=sigmoid,
                 loss_and_delta=binary_cross_entropy):
        self.layer_sizes = list(layer_sizes)
        if len(self.layer_sizes) < 3 or any(size < 1 for size in self.layer_sizes):
            raise ValueError("Use entrada, pelo menos uma camada oculta e saída.")
        self.output_activation = output_activation
        self.loss_and_delta = loss_and_delta
        self.weights = []
        self.biases = []
        for n_in, n_out in zip(self.layer_sizes[:-1], self.layer_sizes[1:]):
            # Inicialização de Xavier: escala adequada para a ativação tanh.
            scale = np.sqrt(2.0 / (n_in + n_out))
            self.weights.append(rng.normal(0.0, scale, size=(n_in, n_out)))
            self.biases.append(np.zeros((1, n_out)))

    def forward(self, X):
        """Amostras nas linhas; guarda as ativações para o backward."""
        activations = [np.asarray(X, dtype=float)]
        for layer, (weights, bias) in enumerate(zip(self.weights, self.biases)):
            logits = activations[-1] @ weights + bias
            if layer == len(self.weights) - 1:
                activation = self.output_activation(logits)
            else:
                activation = np.tanh(logits)
            activations.append(activation)
        return activations[-1], activations, logits

    def backward(self, activations, delta):
        """Deriva todos os pesos e biases antes de alterar os parâmetros."""
        grad_weights = [None] * len(self.weights)
        grad_biases = [None] * len(self.biases)
        for layer in range(len(self.weights) - 1, -1, -1):
            grad_weights[layer] = activations[layer].T @ delta
            grad_biases[layer] = delta.sum(axis=0, keepdims=True)
            if layer > 0:
                # A saída da camada oculta é tanh: derivada local = 1-A².
                delta = (delta @ self.weights[layer].T) * (1.0 - activations[layer] ** 2)
        return grad_weights, grad_biases

    def update(self, grad_weights, grad_biases, learning_rate):
        """Descida do gradiente aplicada a todos os parâmetros."""
        for layer in range(len(self.weights)):
            self.weights[layer] -= learning_rate * grad_weights[layer]
            self.biases[layer] -= learning_rate * grad_biases[layer]

    def fit(self, X, y, learning_rate=0.1, epochs=2000):
        """Treino em lote completo; registra a perda inicial e cada época."""
        history = np.empty(epochs + 1)
        for epoch in range(epochs):
            probabilities, activations, logits = self.forward(X)
            history[epoch], delta = self.loss_and_delta(logits, probabilities, y)
            grad_weights, grad_biases = self.backward(activations, delta)
            self.update(grad_weights, grad_biases, learning_rate)
        probabilities, _, logits = self.forward(X)
        history[-1], _ = self.loss_and_delta(logits, probabilities, y)
        return history

    def predict(self, X):
        probabilities, _, _ = self.forward(X)
        if probabilities.shape[1] == 1:
            return (probabilities[:, 0] >= 0.5).astype(int)
        return np.argmax(probabilities, axis=1)

    @property
    def parameter_count(self):
        return sum(array.size for array in self.weights + self.biases)


In [5]:
layer_sizes = [2, 8, 1]
learning_rate = 0.1
epochs = 2000
model = MLP(layer_sizes, rng)
history = model.fit(X_train, y_train, learning_rate=learning_rate, epochs=epochs)
predictions = model.predict(X_test)
mlp_accuracy = np.mean(predictions == y_test)
print(f"Parâmetros: {model.parameter_count}")
print(f"Perda inicial: {history[0]:.4f}; perda final: {history[-1]:.4f}")
print(f"Acurácia da MLP: {mlp_accuracy:.1%}")
print(f"Acertos: {np.sum(predictions == y_test)} / {len(y_test)}")


Parâmetros: 33
Perda inicial: 0.8270; perda final: 0.1134
Acurácia da MLP: 91.5%
Acertos: 183 / 200


In [6]:
plot_loss(history)
plot_boundary(model, X, X_test, y_test)


![Figura 2 — Perda durante o treinamento](../figures/fig2.png)

![Figura 3 — Fronteira da MLP e pontos de teste](../figures/fig3.png)


### D — Analysis

Na Figura 3, a classe 0 ocupa uma **faixa central**, enquanto a classe 1 ocupa **duas regiões em lados opostos dessa faixa**. A MLP aprendeu duas fronteiras aproximadamente paralelas, com pequenas variações na forma, para manter o grupo central separado dos dois grupos afastados.

Uma única reta divide o plano em dois semiplanos. Ela não consegue colocar os dois grupos opostos da classe 1 de um lado e o grupo central da classe 0 do outro. A camada oculta combina várias transformações não lineares, permitindo atribuir a mesma classe às duas regiões afastadas.

Além disso, a classe 1 foi construída com dois grupos em torno de centros opostos, enquanto a classe 0 fica perto da origem. As médias das classes ficam próximas, e a regressão logística não representa a diferença entre estar no centro e estar afastado em qualquer uma das duas direções. Seu resultado de **47,5%** fica próximo do patamar de 50% de um palpite aleatório em classes balanceadas; o valor abaixo de 50% neste teste é compatível com a variação da amostra e com o ajuste de uma fronteira inadequada para essa estrutura.

A MLP alcançou **91,5%** porque sua fronteira acompanha a organização dos grupos. Os **17 erros** restantes aparecem em um conjunto com sobreposição entre as classes, especialmente próximo às fronteiras. A melhoria foi medida no teste, que não participou do treinamento.


## Results summary

| Quantity | Value |
|---|---|
| Baseline — acurácia no teste | 47,5% |
| MLP — arquitetura | 2 → 8 → 1 |
| MLP — parâmetros | 33 |
| MLP — taxa de aprendizado e épocas | 0,1; 2000 |
| MLP — acurácia no teste | 91,5% (183/200) |
| Perda de treino inicial e final | 0,8270; 0,1134 |
